# Regime A calibration (tuning)

Turn the `WorkloadConfig` knobs until `generate()` output passes the frozen
`spec/regime_A.json` tolerances. `validate()` returns both pass/fail **and** the
MSM objective **Q** to minimize.

Workflow: edit the **Tweak** cell -> re-run it -> watch Q drop -> freeze when seeds pass.

In [ ]:
import dataclasses
from pathlib import Path
import numpy as np
import workload_gen as wg

# Robust to cwd: locate the frozen spec next to the package, not via a relative path.
SPEC = Path(wg.__file__).parent / "spec" / "regime_A.json"

def evaluate(cfg, seeds=range(8), show=True):
    """Generate + validate over several seeds. Reports per-check pass-rate with
    current mean vs spec target, plus TWO objectives:
      Q_feasible  -> distance OUTSIDE the tolerances; 0 when all pass. MINIMIZE THIS.
      Q_faithful  -> distance to the real-day point targets (how realistic the trace is).
    Judge a setting on the pass-rate across seeds, NEVER one lucky draw (seed 0).
    Returns (mean Q_feasible, reports)."""
    reports = [wg.validate(wg.generate(cfg, seed=s), SPEC) for s in seeds]
    Qf = [r.distance_feasible for r in reports]
    Qt = [r.distance_faithful for r in reports]
    n = len(reports)
    if show:
        print(f"per-check over {n} seeds   (current mean vs spec target):")
        for i, c in enumerate(reports[0].checks):
            rate = sum(r.checks[i].passed for r in reports)
            cur = np.mean([r.checks[i].value for r in reports])
            flag = "[pass]" if rate == n else "[fail]"
            print(f"  {flag} {c.name:<22} cur {cur:< 11.4g} target {c.target:< 11.4g}  {rate}/{n}")
        n_all = sum(r.passed for r in reports)
        print(f"mean Q_feasible = {np.mean(Qf):.4g}   (search objective: -> 0 when all pass)")
        print(f"mean Q_faithful = {np.mean(Qt):.4g}   (distance to real-day point targets)")
        print(f"all-checks-pass: {n_all}/{n}")
    return float(np.mean(Qf)), reports

## Knob reference — every tunable knob

**Core model** (per-rack power, watts):
`power_r(t) = scale_r * ( floor + Sum_j p_j * profile_j(t) * 1[r in S_j] * 1[t in active_j] ) + noise`

**Key relations** (M/G/inf occupancy; these explain the formulas below):
- occupancy (mean # active jobs on a rack) = `lambda * E[frac] * E[d]`
- `E[power_r] = scale_r * (floor + occupancy * E[p_j])`  -> this is what sets per_rack_mean & total_mean
- lognormal mean `E[d] = exp(mu + sigma^2/2)`  ;  beta mean `E[frac] = a/(a+b)`

| Knob (cfg field) | Represents | Formula / how set | Impacts which validation stats | Bucket |
|---|---|---|---|---|
| `idle_floor_W` | per-rack idle baseline power | from spec ~253 kW; added to every cell before jobs | marginal **idle mode** (~0.25 MW); additive floor in per_rack_mean & total_mean | A |
| `per_rack_scale[r]` | per-rack hardware scale (homogeneity) | `scale_r = mean_r / grand_mean` (~1; rack14=0.652); applied `power[:,r] *= scale_r` | **per_rack_mean** shape across racks (relative levels) | A |
| `job_power` (`mean`,`std`) | power one job adds per rack (shared by the job's racks) | `mean = busy - floor ~= 756 kW`; `p_j ~ Normal(mean,std)` | marginal **busy mode** height (~1.0 MW); per_rack_mean/total_mean via `occ*E[p]`; `std` = busy-mode width | A |
| `arrival_rate_per_s` (lambda) | Poisson job arrival rate (jobs/s) | `N ~ Poisson(lambda*(T+burn))`; seeded `lambda = occupancy/(E[frac]*E[d])`, `occupancy=(grand_mean-floor)/job_power_mean` | **level**: per_rack_mean & total_mean; busy-fraction; peak stacking; ramp density | A |
| `duration` (`mu`,`sigma`) | job length, seconds (persistence) | `d_j ~ Lognormal(mu,sigma)`; `mu` set so `E[d]=tau*dt=2610 s` | `E[d]` -> **autocorr tau** & occupancy(level); **`sigma` (tail)** -> per_rack_mean sampling variance + ramp **kurtosis** | A (E[d]) / B (sigma) |
| `job_size` (`a`,`b`) | job size as a **fraction of the machine** -> k racks | `frac ~ Beta(a,b)`, `E=a/(a+b)`; `k = max(1, round(frac*n_racks))` | **offdiag_corr & pc1_var_share** (synchronization via co-placement); also occupancy via `E[frac]` (level) | C |
| `placement` | which k racks a job lands on | `"scattered"` = random subset; `"contiguous"` = consecutive block (wrap) | residual **off-diagonal block structure** (real day = none -> scattered); minor on aggregates | C |
| `noise_amp_W` | std of additive per-rack noise per step | `power += Normal(0, noise_amp_W)`, then clip >=0 | **up** ramp_abs_mean, **down** kurtosis; **but down** corr/pc1 (rack-independent variance) -> tradeoff | A |
| `burstiness` | arrival clustering beyond Poisson (reserved) | no-op at `0.0` (pure Poisson); future Hawkes / neg-binomial | (future) ramp & job-boundary burstiness; none while 0 | B |
| `_job_profile` *(generator.py, NOT a cfg field)* | intra-job power shape over active steps | flat `np.ones(n_active)`; editable to ramp-up/down or correlated wiggle | **up** ramp_abs_mean, **down** kurtosis **without** killing corr (variation shared across the job's racks -> stays synchronized) | code |

**Notes**
- **noise vs `_job_profile`:** raw `noise_amp_W` fixes ramps/kurtosis but erodes corr; a shaped `_job_profile` adds within-job ramps that move *together* across a job's racks -> fixes ramps **without** lowering corr. Prefer the profile.
- **Coupling:** `arrival_rate` and `job_size` BOTH move occupancy (`lambda*E[frac]*E[d]`). If you change `job_size`'s mean, re-derive `arrival_rate` to keep the level, or the mean drifts.
- **per_rack_mean residual** (after the occupancy fix) is sampling noise from few/heavy-tailed jobs -> lower `duration.sigma`; always judge on the **multi-seed** pass-rate, never seed 0.
- **Not knobs:** `n_racks`, `n_steps`, `dt`, `seed` are `generate()` args (output shape + RNG), deliberately kept out of the config.

In [9]:
# Baseline: the constraint-derived starting theta
cfg = wg.WorkloadConfig.regime_A_starting(SPEC)
evaluate(cfg);

Validation: FAIL   Q(theta) = 18.64
  [FAIL] per_rack_mean          =  5.2377       (max |dev| 5.2% <= 5%)
  [PASS] total_mean_W           =  1.6768e+07   (16.77 MW vs 16.12 +/-7%)
  [PASS] pc1_var_share          =  0.98027      (>= 0.98)
  [PASS] ramp_excess_kurtosis   =  133.87       (>= 15)
  [PASS] offdiag_corr_mean      =  0.97782      (in [0.95, 0.999])
mean Q over 5 seeds = 23.21 (min 5.92, max 40.9) | passing: 0/5


### Tuning
- **Keep seeds fixed** while turning one knob, so Q changes only from your edit (common random numbers).
- A setting is "passing" only if it passes across **multiple seeds**.
- Read the **dominant squared term** in Q to choose the next knob.
- Knobs **interact** (occupancy moves both total_mean and busy-fraction) -- adjust one at a time.

In [10]:
# === TWEAK CELL: edit knobs, re-run, watch Q ===
cfg = wg.WorkloadConfig.regime_A_starting(SPEC)        # start fresh (comment out to keep tuning the same cfg)

cfg = dataclasses.replace(                             # replace() rebuilds -> re-runs validation
    cfg,
    noise_amp_W = 30_000.0,
    # arrival_rate_per_s = 3.0e-4,
    # job_power = wg.DistSpec("normal", {"mean": 756_000, "std": 50_000}),
    # duration  = wg.DistSpec("lognormal", {"mu": 7.62, "sigma": 1.0}),
    # job_size  = wg.DistSpec("beta", {"a": 30, "b": 1.5}),
    # placement = "contiguous",
)
evaluate(cfg);

Validation: FAIL   Q(theta) = 2.694
  [FAIL] per_rack_mean          =  5.3641       (max |dev| 5.4% <= 5%)
  [PASS] total_mean_W           =  1.6767e+07   (16.77 MW vs 16.12 +/-7%)
  [FAIL] pc1_var_share          =  0.97718      (>= 0.98)
  [PASS] ramp_excess_kurtosis   =  66.478       (>= 15)
  [PASS] offdiag_corr_mean      =  0.97448      (in [0.95, 0.999])
mean Q over 5 seeds = 3.231 (min 1.76, max 5.04) | passing: 0/5


In [ ]:
# Freeze the calibrated config once seeds pass
out = Path(wg.__file__).parent / "spec" / "regime_A_calib.json"
cfg.to_json(out)
print("saved", out)